# Visual Similarity Pipeline — Canonical Notebook (Ground Zero)

**What this is:** the single, leak-proof pipeline that everything else forks from. It runs end to end: load features, split by image (leak-free), engineer pair features, race models, compute the noise ceiling, and evaluate against consensus human similarity.

**The throughline:** *Can deep vision features (CLIP/DINO) predict human similarity judgments, and how much of human similarity is recoverable from perception alone?*

**Honest headline (full data):** consensus **r ≈ 0.38** (95% CI [0.31, 0.44]) against a **noise ceiling of 0.75** — about **half** of the achievable structure. A naive pair-level split had inflated this to 0.34 via leakage; the image-disjoint split below is the fix.

> Runs in **DEMO_MODE** on a small bundled sample so you can see it execute. Set `DEMO_MODE=False` and point paths at the full data to reproduce the headline.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────
DEMO_MODE = True          # True: run on sample_data/  |  False: full data on Drive
SEED      = 42
FIT_CAP   = 80_000        # cap training rows for model fits (linear models plateau early; keeps RAM ~2GB)
SUBSAMPLE = 3_000 if DEMO_MODE else 40_000   # demo: small & fast | full data: 40k for selection
N_STRATA  = 10

import os, numpy as np, pandas as pd
from scipy.stats import pearsonr
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import RidgeCV, LassoCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
rng = np.random.default_rng(SEED)

if DEMO_MODE:
    BASE = "sample_data/"
    CLIP, DINO, PAIRS = BASE+"clip_sample.csv", BASE+"dino_sample.csv", BASE+"pairs_sample.csv"
else:
    BASE = "/content/drive/MyDrive/visual-similarity-data/"
    CLIP, DINO, PAIRS = BASE+"Features/konkle_clip_features.csv", BASE+"Features/konkle_dino_features.csv", BASE+"Features/konkle_similarity_pairs.csv"
print("DEMO_MODE =", DEMO_MODE)

## 1. Load features (CLIP + DINO), merge on image basename

In [ ]:
clip_df = pd.read_csv(CLIP); dino_df = pd.read_csv(DINO)
cc = [x for x in clip_df.columns if x.startswith("clip_")]
dc = [x for x in dino_df.columns if x.startswith("dino_")]
merged = clip_df[["basename"]+cc].merge(dino_df[["basename"]+dc], on="basename")
feats = merged[cc+dc].to_numpy(dtype=np.float32)          # (n_images, 896)
idx = {b:i for i,b in enumerate(merged["basename"].astype(str))}
print(f"Images with both features: {len(merged)}   feature dims: {feats.shape[1]}")

## 2. Leak-proof split (image-disjoint)

Split the **images** 80/20, then keep a pair only if **both** its images are on the same side. This is what prevents an image the model trained on from reappearing in the test set. Target normalization is fit on **train only** (no pooled leakage).

In [ ]:
pairs = pd.read_csv(PAIRS)
pairs["image1"] = pairs["image1"].astype(str); pairs["image2"] = pairs["image2"].astype(str)
feat_imgs = set(idx)
pairs = pairs[pairs["image1"].isin(feat_imgs) & pairs["image2"].isin(feat_imgs)].reset_index(drop=True)

imgs = sorted(set(pairs["image1"]) | set(pairs["image2"]))
perm = rng.permutation(len(imgs)); n_test = int(round(0.2*len(imgs)))
test_imgs  = {imgs[i] for i in perm[:n_test]}
train_imgs = {imgs[i] for i in perm[n_test:]}
tr_mask = pairs["image1"].isin(train_imgs) & pairs["image2"].isin(train_imgs)
te_mask = pairs["image1"].isin(test_imgs)  & pairs["image2"].isin(test_imgs)
train_pairs, test_pairs = pairs[tr_mask].copy(), pairs[te_mask].copy()

overlap = (set(train_pairs.image1)|set(train_pairs.image2)) & (set(test_pairs.image1)|set(test_pairs.image2))
print(f"Train {len(train_pairs):,} | Test {len(test_pairs):,} | dropped {int((~tr_mask&~te_mask).sum()):,} ({100*(~tr_mask&~te_mask).mean():.1f}%)")
print(f"IMAGE OVERLAP: {len(overlap)}  {'OK leak-proof' if not overlap else 'LEAK!'}")

# target: invert (higher = more similar), fit min/max on TRAIN only
ytr = train_pairs["similarity"].to_numpy(float); yte = test_pairs["similarity"].to_numpy(float)
lo, span = ytr.min(), (ytr.max()-ytr.min()) or 1.0
train_pairs["y"] = 1 - (ytr-lo)/span; test_pairs["y"] = 1 - (yte-lo)/span

## 3. Engineer pair features (abs-diff | product | cosine = 2,688d)

In [ ]:
def engineer(df):
    a = df["image1"].map(idx).to_numpy(); b = df["image2"].map(idx).to_numpy()
    f1, f2 = feats[a], feats[b]
    ad, pr = np.abs(f1-f2), f1*f2
    den = np.linalg.norm(f1,axis=1)*np.linalg.norm(f2,axis=1)+1e-10
    cos = np.repeat((np.sum(f1*f2,axis=1)/den).astype(np.float32)[:,None], f1.shape[1], 1)
    return np.concatenate([ad,pr,cos],1).astype(np.float32), df["y"].to_numpy(float)

## 4. Fair model race (selection only)

Same stratified subsample and same test set for all five. Linear models get `StandardScaler` (fit on train only); trees run raw. This gives each model its proper input form, no thumb on the scale.

In [ ]:
if len(train_pairs) <= SUBSAMPLE:
    sub = train_pairs                      # small data (demo): use it all
else:
    y_all = train_pairs["y"].to_numpy()
    bins = np.clip((y_all*N_STRATA).astype(int),0,N_STRATA-1)
    frac = SUBSAMPLE/len(train_pairs)
    sel = np.concatenate([rng.choice(np.where(bins==k)[0], int(round((bins==k).sum()*frac)), replace=False)
                          for k in range(N_STRATA) if (bins==k).sum()>0])
    sub = train_pairs.iloc[sel]
Xs, ys = engineer(sub); Xt, yt = engineer(test_pairs)
sc = StandardScaler().fit(Xs); Xs_s, Xt_s = sc.transform(Xs), sc.transform(Xt)

racers = [("Random Forest", RandomForestRegressor(n_estimators=50,max_depth=20,random_state=SEED,n_jobs=-1), False),
          ("Gradient Boosting", GradientBoostingRegressor(n_estimators=100,max_depth=3,random_state=SEED), False),
          ("Ridge", RidgeCV(alphas=np.logspace(-2,4,13)), True),
          ("Lasso", LassoCV(alphas=np.logspace(-4,1,10),random_state=SEED,n_jobs=-1,max_iter=5000), True),
          ("Linear", LinearRegression(), True)]
rows=[]
for name,m,scaled in racers:
    Xtr,Xte = (Xs_s,Xt_s) if scaled else (Xs,Xt)
    m.fit(Xtr,ys); r = pearsonr(yt, m.predict(Xte))[0]
    rows.append((name,r)); print(f"  {name:18s} r = {r:.4f}")
race = pd.DataFrame(rows,columns=["Model","Honest r"]).sort_values("Honest r",ascending=False)
print("\nWinner:", race.iloc[0].Model)
del Xs,Xt,Xs_s,Xt_s

## 5. Noise ceiling (split-half, Spearman-Brown)

Each pair was judged ~19 times. Split those judgments in half, correlate the half-means across pairs, Spearman-Brown correct. This is the best correlation ANY model predicting the mean could reach.

In [ ]:
g = test_pairs.groupby(["image1","image2"])["y"].apply(list)
multi = [v for v in g if len(v)>=2]
rs=[]
for _ in range(50):
    A,B=[],[]
    for v in multi:
        v=np.array(v); rng.shuffle(v); h=len(v)//2
        if h: A.append(v[:h].mean()); B.append(v[h:2*h].mean())
    rh=pearsonr(A,B)[0]; rs.append(2*rh/(1+rh))
ceiling=float(np.mean(rs))
print(f"Noise ceiling: {ceiling:.4f}  (n pairs with repeats: {len(multi)})")

## 6. Final evaluation: consensus r vs the ceiling

Fit the sensible winner (Ridge) on a capped sample, then score against the **consensus** (pair mean), not individual judgments. Report as a fraction of the ceiling.

In [ ]:
fit_df = train_pairs.sample(min(FIT_CAP,len(train_pairs)), random_state=SEED)
Xf, yf = engineer(fit_df); Xte, yte2 = engineer(test_pairs)
scf = StandardScaler().fit(Xf)
model = RidgeCV(alphas=np.logspace(-2,4,13)).fit(scf.transform(Xf), yf)
pred = model.predict(scf.transform(Xte))
del Xf, Xte

r_ind = pearsonr(yte2, pred)[0]
te = test_pairs.copy(); te["pred"]=pred
agg = te.groupby(["image1","image2"]).agg(y=("y","mean"),p=("pred","mean")).reset_index()
r_con = pearsonr(agg.y, agg.p)[0]
boot=[pearsonr(agg.y.values[s],agg.p.values[s])[0] for s in (rng.integers(0,len(agg),len(agg)) for _ in range(2000))]
lo_ci,hi_ci = np.percentile(boot,[2.5,97.5])
print(f"vs individual : r = {r_ind:.4f}")
print(f"vs CONSENSUS  : r = {r_con:.4f}  95% CI [{lo_ci:.3f}, {hi_ci:.3f}]  (n={len(agg)})")
print(f"NOISE CEILING : r = {ceiling:.4f}")
print(f"--> {100*r_con/ceiling:.0f}% of achievable ceiling")

## Interpretation

- Ceiling high (~0.75 on full data) means humans genuinely agree; there is a real consensus target.
- Model reaches ~half of it. The captured half is largely linear/perceptual; the missing half plausibly reflects categorical/semantic structure (the perception/categorization gap).
- Fit for **structure/triage** use, not exact-pairwise ground truth.
